In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

catalog, schema = "workspace", "patient_notes"
bronze_table = f"{catalog}.{schema}.bronze_notes"
silver_table = f"{catalog}.{schema}.silver_notes"
alias_table  = f"{catalog}.{schema}.ref_condition_aliases"

In [0]:
aliases = [
    ("pneumonia", "pneumonia"), ("pna", "pneumonia"),
    ("type 2 diabetes", "type 2 diabetes"), ("t2dm", "type 2 diabetes"), ("dm2", "type 2 diabetes"),
    ("diabetes mellitus type 2", "type 2 diabetes"),
    ("hypertension", "hypertension"), ("htn", "hypertension"), ("high blood pressure", "hypertension"),
    ("asthma", "asthma"),
    ("lung tumor", "lung tumor"), ("lung mass", "lung tumor"), ("pulmonary neoplasm", "lung tumor"),
    ("chronic kidney disease", "chronic kidney disease"), ("ckd", "chronic kidney disease"),
    ("malaria", "malaria"),
    ("urinary tract infection", "urinary tract infection"), ("uti", "urinary tract infection"),
    ("migraine", "migraine"),
    ("copd", "copd"), ("chronic obstructive pulmonary disease", "copd"),
    ("fatty liver", "fatty liver"), ("hepatic steatosis", "fatty liver"), ("nafld", "fatty liver"),("appendicitis", "appendicitis"), ("acute appendicitis", "appendicitis"),
("gallstones", "gallstones"), ("cholelithiasis", "gallstones"),
]
(spark.createDataFrame(aliases, "alias string, standard_name string")
      .write.mode("overwrite").saveAsTable(alias_table))
display(spark.table(alias_table))

In [0]:
bronze = spark.table(bronze_table)

latest_first = Window.partitionBy("note_id").orderBy(F.col("ingested_at").desc())

clean = (bronze
    .withColumn("note_date", F.to_date("note_date"))          # string -> real date
    .withColumn("note_text", F.trim("note_text"))
    .withColumn("rn", F.row_number().over(latest_first))      # 1 = newest copy of each note
    .filter("rn = 1")
    .drop("rn", "_rescued_data"))

print("Bronze rows:", bronze.count(), "| After dedup:", clean.count())

In [0]:
sections = (clean
    .withColumn("chief_complaint", F.regexp_extract("note_text", r"Chief complaint:\s*([^\n]*)", 1))
    .withColumn("assessment",      F.regexp_extract("note_text", r"(?s)Assessment:\s*(.*?)\s*Plan:", 1))
    .withColumn("plan",            F.regexp_extract("note_text", r"(?s)Plan:\s*(.*)", 1)))

display(sections.select("note_id", "note_type", "chief_complaint", "assessment", "plan").limit(5))

In [0]:
aliases_df = spark.table(alias_table)

mentions = (sections.select("note_id", "assessment")
    .crossJoin(F.broadcast(aliases_df))                         # every note x every alias (small table)
    .filter(F.expr(r"lower(assessment) rlike concat('\\b', alias, '\\b')"))   # whole-word match
    .groupBy("note_id")
    .agg(F.sort_array(F.collect_set("standard_name")).alias("conditions_mentioned")))

silver_df = sections.join(mentions, "note_id", "left")
display(silver_df.select("note_id", "assessment", "conditions_mentioned").limit(10))

In [0]:
if not spark.catalog.tableExists(silver_table):
    silver_df.limit(0).write.saveAsTable(silver_table)          # create an empty table with the right columns

(DeltaTable.forName(spark, silver_table).alias("t")
    .merge(silver_df.alias("s"), "t.note_id = s.note_id")
    .whenMatchedUpdateAll()                                     # existing note -> update
    .whenNotMatchedInsertAll()                                  # new note -> insert
    .execute())

print("Silver rows:", spark.table(silver_table).count())